# Risk Action Recommender Training


Train the action recommender that maps fraud/cyber/behavior scores to an operational action (ALLOW / MONITOR / REVIEW / BLOCK).


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
for cand in [repo_root, *repo_root.parents]:
    if (cand / 'app').exists():
        repo_root = cand
        break
sys.path.insert(0, str(repo_root))
print('Repo root:', repo_root)


## Data requirements


Uses existing score exports under `experiments/`. If `experiments/fusion/fusion_scores.csv` is present, it will be used. Otherwise it falls back to `experiments/fraud/scores.csv` or `experiments/cyber/scores.csv`.


## Links to code


- Training script: `src/train/train_recommender.py`


- Risk engine: `app/services/risk_engine.py`


- API endpoints: `POST /api/risk/analyze` and `POST /api/fraud`


## Dataset inventory


In [ ]:
from pathlib import Path

fusion_csv = repo_root / 'experiments' / 'fusion' / 'fusion_scores.csv'
fraud_csv = repo_root / 'experiments' / 'fraud' / 'scores.csv'
cyber_csv = repo_root / 'experiments' / 'cyber' / 'scores.csv'
print('Fusion scores:', fusion_csv.exists(), fusion_csv)
print('Fraud scores:', fraud_csv.exists(), fraud_csv)
print('Cyber scores:', cyber_csv.exists(), cyber_csv)


## Train the model


In [ ]:
# !python -m src.train.train_recommender


## Visuals and metrics


In [ ]:
import json
import matplotlib.pyplot as plt

metrics_path = repo_root / 'experiments' / 'recommender' / 'metrics' / 'metrics.json'
if not metrics_path.exists():
    print('Metrics not found:', metrics_path)
else:
    report = json.loads(metrics_path.read_text(encoding='utf-8'))
    labels = [k for k in report.keys() if k not in {'accuracy', 'macro avg', 'weighted avg'}]
    if not labels:
        print('No class labels in report')
    else:
        f1s = [report[l]['f1-score'] for l in labels if isinstance(report.get(l), dict)]
        plt.figure(figsize=(6, 3))
        plt.bar(labels[: len(f1s)], f1s)
        plt.title('Action recommender F1 scores')
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()
